In [ ]:
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [8]:
DATA_URL = "https://raw.githubusercontent.com/Mahin2903/220148_AI-ML_Lab/main/Churn_Modelling.csv"

df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
drop_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
df = df.drop(columns=drop_cols)

TARGET = "Exited"
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Missing values per column:")
print(X.isnull().sum())

Missing values per column:
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
dtype: int64


In [10]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Step 1: Imputation (defensive — handles missing data if/when present)
num_imputer = SimpleImputer(strategy="median")
X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])

if categorical_cols:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical columns: ['Geography', 'Gender']


In [11]:
# Step 2: Stratified split -> Train / Val / Test (80/20, then 80/20 of train for validation)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)

# Step 3: Encode categoricals (One-Hot) + scale numerics — fit ONLY on training data
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical_cols),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

X_train = np.asarray(X_train, dtype=np.float32)
X_val = np.asarray(X_val, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

y_train = y_train.values.astype(np.float32)
y_val = y_val.values.astype(np.float32)
y_test = y_test.values.astype(np.float32)

input_dim = X_train.shape[1]
print("Train/Val/Test shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Input feature dimension after encoding:", input_dim)

Train/Val/Test shapes: (6400, 12) (1600, 12) (2000, 12)
Input feature dimension after encoding: 12


In [12]:
def make_loader(X, y, batch_size, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

X_val_t = torch.tensor(X_val).to(device)
y_val_t = torch.tensor(y_val).unsqueeze(1).to(device)
X_test_t = torch.tensor(X_test).to(device)
y_test_t = torch.tensor(y_test).unsqueeze(1).to(device)

In [13]:
class ShallowNN(nn.Module):
    def __init__(self, input_dim, hidden_units=32, activation="relu"):
        super().__init__()
        act_layer = nn.ReLU() if activation == "relu" else nn.Sigmoid()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            act_layer,
            nn.Linear(hidden_units, 1),
        )

    def forward(self, x):
        return self.net(x)

In [14]:
def train_model(model, train_loader, X_val_t, y_val_t, epochs, lr,
                 optimizer_name="adam", weight_decay=0.0):
    criterion = nn.BCEWithLogitsLoss()
    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == yb).sum().item()
            total += xb.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_preds = (torch.sigmoid(val_logits) >= 0.5).float()
            val_acc = (val_preds == y_val_t).float().mean().item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    return history


@torch.no_grad()
def evaluate_model(model, X_t, y_t):
    model.eval()
    logits = model(X_t)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    preds = (probs >= 0.5).astype(int)
    y_true = y_t.cpu().numpy().ravel().astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "auc": roc_auc_score(y_true, probs),
    }
    return metrics, probs, preds, y_true

In [ ]:
hidden_units_grid = [16, 32, 64]
activation_grid = ["relu", "sigmoid"]
batch_size_grid = [16, 32, 64]
TUNE_EPOCHS = 20

best_shallow_cfg, best_shallow_val_acc = None, -1
shallow_tuning_results = []

for hu in hidden_units_grid:
    for act in activation_grid:
        for bs in batch_size_grid:
            torch.manual_seed(SEED)
            model = ShallowNN(input_dim, hidden_units=hu, activation=act).to(device)
            loader = make_loader(X_train, y_train, batch_size=bs)
            hist = train_model(model, loader, X_val_t, y_val_t,
                                epochs=TUNE_EPOCHS, lr=1e-3, optimizer_name="adam")
            val_acc = hist["val_acc"][-1]
            shallow_tuning_results.append(
                {"hidden_units": hu, "activation": act, "batch_size": bs, "val_acc": val_acc}
            )
            if val_acc > best_shallow_val_acc:
                best_shallow_val_acc = val_acc
                best_shallow_cfg = {"hidden_units": hu, "activation": act, "batch_size": bs}

print("Best Shallow NN config:", best_shallow_cfg, "| Val Acc:", round(best_shallow_val_acc, 4))
pd.DataFrame(shallow_tuning_results).sort_values("val_acc", ascending=False).head()

In [ ]:
SHALLOW_EPOCHS = 60
torch.manual_seed(SEED)
shallow_model = ShallowNN(
    input_dim,
    hidden_units=best_shallow_cfg["hidden_units"],
    activation=best_shallow_cfg["activation"],
).to(device)
shallow_loader = make_loader(X_train, y_train, batch_size=best_shallow_cfg["batch_size"])
shallow_history = train_model(
    shallow_model, shallow_loader, X_val_t, y_val_t,
    epochs=SHALLOW_EPOCHS, lr=1e-3, optimizer_name="adam"
)
print("Final Shallow NN val acc:", round(shallow_history["val_acc"][-1], 4))

In [ ]:
class DeepNN(nn.Module):
    def __init__(self, input_dim, hidden_layers=(128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
lr_grid = [1e-2, 1e-3, 1e-4]
optimizer_grid = ["adam", "sgd"]
epochs_grid = [30, 60]
TUNE_BATCH_SIZE = 32

best_deep_cfg, best_deep_val_acc = None, -1
deep_tuning_results = []

for lr in lr_grid:
    for opt_name in optimizer_grid:
        for ep in epochs_grid:
            torch.manual_seed(SEED)
            model = DeepNN(input_dim, hidden_layers=(128, 64, 32), dropout=0.3).to(device)
            loader = make_loader(X_train, y_train, batch_size=TUNE_BATCH_SIZE)
            hist = train_model(model, loader, X_val_t, y_val_t,
                                epochs=ep, lr=lr, optimizer_name=opt_name, weight_decay=1e-4)
            val_acc = hist["val_acc"][-1]
            deep_tuning_results.append(
                {"lr": lr, "optimizer": opt_name, "epochs": ep, "val_acc": val_acc}
            )
            if val_acc > best_deep_val_acc:
                best_deep_val_acc = val_acc
                best_deep_cfg = {"lr": lr, "optimizer": opt_name, "epochs": ep}

print("Best Deep NN config:", best_deep_cfg, "| Val Acc:", round(best_deep_val_acc, 4))
pd.DataFrame(deep_tuning_results).sort_values("val_acc", ascending=False).head()

In [ ]:
torch.manual_seed(SEED)
deep_model = DeepNN(input_dim, hidden_layers=(128, 64, 32), dropout=0.3).to(device)
deep_loader = make_loader(X_train, y_train, batch_size=TUNE_BATCH_SIZE)
deep_history = train_model(
    deep_model, deep_loader, X_val_t, y_val_t,
    epochs=best_deep_cfg["epochs"], lr=best_deep_cfg["lr"],
    optimizer_name=best_deep_cfg["optimizer"], weight_decay=1e-4,
)
print("Final Deep NN val acc:", round(deep_history["val_acc"][-1], 4))

In [ ]:
shallow_metrics, shallow_probs, shallow_preds, y_true = evaluate_model(shallow_model, X_test_t, y_test_t)
deep_metrics, deep_probs, deep_preds, _ = evaluate_model(deep_model, X_test_t, y_test_t)

print("Shallow NN Test Metrics:", {k: round(v, 4) for k, v in shallow_metrics.items()})
print("Deep NN Test Metrics:   ", {k: round(v, 4) for k, v in deep_metrics.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(shallow_history["train_loss"], label="Train Loss")
axes[0].plot(shallow_history["val_loss"], label="Val Loss")
axes[0].set_title("Shallow NN — Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

axes[1].plot(deep_history["train_loss"], label="Train Loss")
axes[1].plot(deep_history["val_loss"], label="Val Loss")
axes[1].set_title("Deep NN — Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(shallow_history["train_acc"], label="Train Acc")
axes[0].plot(shallow_history["val_acc"], label="Val Acc")
axes[0].set_title("Shallow NN — Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

axes[1].plot(deep_history["train_acc"], label="Train Acc")
axes[1].plot(deep_history["val_acc"], label="Val Acc")
axes[1].set_title("Deep NN — Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
shallow_cm = confusion_matrix(y_true, shallow_preds)
deep_cm = confusion_matrix(y_true, deep_preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(shallow_cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Shallow NN — Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

sns.heatmap(deep_cm, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Deep NN — Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
shallow_fpr, shallow_tpr, _ = roc_curve(y_true, shallow_probs)
deep_fpr, deep_tpr, _ = roc_curve(y_true, deep_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(shallow_fpr, shallow_tpr, label=f"AUC = {shallow_metrics['auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("Shallow NN — ROC Curve")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate"); axes[0].legend()

axes[1].plot(deep_fpr, deep_tpr, label=f"AUC = {deep_metrics['auc']:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("Deep NN — ROC Curve")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate"); axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Shallow NN AUC: {shallow_metrics['auc']:.4f}")
print(f"Deep NN AUC:    {deep_metrics['auc']:.4f}")

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "auc"]
shallow_vals = [shallow_metrics[m] for m in metric_names]
deep_vals = [deep_metrics[m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, shallow_vals, width, label="Shallow NN")
ax.bar(x + width/2, deep_vals, width, label="Deep NN")

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metric_names])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Shallow vs. Deep NN — Evaluation Metrics")
ax.legend()

for i, v in enumerate(shallow_vals):
    ax.text(i - width/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
for i, v in enumerate(deep_vals):
    ax.text(i + width/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q torchinfo
from torchinfo import summary

print("=" * 60)
print("SHALLOW NEURAL NETWORK ARCHITECTURE")
print("=" * 60)
summary(shallow_model, input_size=(1, input_dim))

print()
print("=" * 60)
print("DEEP NEURAL NETWORK ARCHITECTURE")
print("=" * 60)
summary(deep_model, input_size=(1, input_dim))